<a href="https://colab.research.google.com/github/Otza02/satellite-img-segmentation/blob/main/notebooks/cross-val.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Otza02/satellite-img-segmentation.git
%cd satellite-img-segmentation
%pip install -e .

## Reiniciar sesion

In [ ]:
%cd /content/satellite-img-segmentation
!unzip -q data/train.zip -d data/train

## Cross Validation

In [ ]:
from satelliteSegmentation.dataset import AugmentedSubset, SatelliteData, kfold_loaders
from satelliteSegmentation.train import train_model
from satelliteSegmentation.config import Config
from satelliteSegmentation.metrics import segmentation_metrics
from satelliteSegmentation.models import UNet

from torchvision.transforms import v2

import torch
from torch.utils.data import DataLoader

import json
from pathlib import Path

In [ ]:
def calc_weights(
    dataloader,
    num_classes: int,
    ignore_index: int,
):
    counts = torch.zeros(num_classes, dtype=torch.long)

    for _, masks in dataloader:
        masks = masks.flatten()
        masks = masks[masks != ignore_index]

        counts += torch.bincount(
            masks,
            minlength=num_classes,
        )

    # Solo clases que participan en el entrenamiento
    valid_counts = counts.clone()
    valid_counts[ignore_index] = 0

    # Ejemplo: inverse square root frequency
    weights = torch.zeros(num_classes, dtype=torch.float32)

    valid = valid_counts > 0
    weights[valid] = 1.0 / torch.sqrt(valid_counts[valid].float())

    # Normalizar para que la media de los weights válidos sea 1
    weights[valid] /= weights[valid].mean()

    # El weight de la clase ignorada es irrelevante
    weights[ignore_index] = 0.0

    return counts, weights

# counts, weights = calc_weights(
#     train_loader,
#     num_classes=7,
#     ignore_index=6,
# )
# weights = weights.to(device)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

In [ ]:
data = SatelliteData("data/train")
shape_tf = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomRotation([0, 180], fill=6),
])

color_tf = v2.Compose([
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1)
])

In [ ]:
loader_kwargs = {
    "num_workers": 2,
    "pin_memory": device == "cuda",
    "persistent_workers": device == "cuda",
}

k = 5
results = {}
for fold, train_loader, val_loader in kfold_loaders(
    data,
    k=k,
    batch_size=batch_size,
    shape_transform=shape_tf,
    color_transform=color_tf,
    **loader_kwargs,
):
    _, weights = calc_weights(
        train_loader,
        num_classes=7,
        ignore_index=6,
    )
    weights = weights.to(device)
    config = Config(device, weight_decay=2e-5, kernel_size=3, lr=1.51e-04, weights=weights, patience=15)
    print(f"=== Fold {fold + 1}/{k} ===")
    model = UNet(config)
    criterion = torch.nn.CrossEntropyLoss(config.weights, ignore_index=6)
    model, hist = train_model(model, train_loader, val_loader, criterion, config)
    metrics = segmentation_metrics(model, val_loader, config.device, config.num_classes)

    results[f"fold={fold}"] = {
        "hist": hist,
        "metrics": metrics,
    }

In [ ]:
from copy import deepcopy

results_backup = deepcopy(results)

In [ ]:
results[f"fold=0"]["metrics"]["confusion_matrix"].tolist()

for fold in results.keys():
    results[fold]["metrics"]["confusion_matrix"] = results[fold]["metrics"]["confusion_matrix"].tolist()

In [ ]:
result_folder = Path("result/cross-val")
result_folder.mkdir(parents=True, exist_ok=True)
with open(result_folder / "models.json", "w") as f:
    json.dump(results, f)